# 05 — Visualización y verificación del resultado final
Consultas sobre las tablas Gold para verificar e interpretar el resultado. No forma parte del Job.
Para graficar: en la salida de cada celda, **+ → Visualization** (tipo sugerido en cada título).

## 1. Ventaja de local según el régimen de público
Gráfico sugerido: barras agrupadas, X = `liga`, Y = `pct_local` y `prob_local_mercado`, agrupado por `regimen_publico`.

In [0]:
SELECT liga, regimen_publico, visitantes_permitidos, partidos,
       round(pct_local * 100, 1)           AS pct_local,
       round(prob_local_mercado * 100, 1)  AS prob_local_mercado,
       diferencia_goles_promedio
FROM workspace.tp_futbol.gold_resumen_regimen_publico
ORDER BY liga, regimen_publico, visitantes_permitidos;

liga,regimen_publico,visitantes_permitidos,partidos,pct_local,prob_local_mercado,diferencia_goles_promedio
ARG,parcial,no,91,44.0,42.1,0.297
ARG,pleno,no,5495,43.5,42.8,0.304
ARG,pleno,si,363,41.1,42.4,0.331
ARG,sin_publico,na,479,38.6,40.3,0.157
D1,parcial,si,290,46.9,44.3,0.455
D1,pleno,si,2180,43.9,45.0,0.359
D1,sin_publico,na,338,40.5,41.8,0.175
E0,parcial,no,77,41.6,42.2,0.169
E0,pleno,si,3018,44.7,44.3,0.292
E0,sin_publico,na,395,39.2,41.1,0.063


## 2. Error del mercado sobre el local en partidos sin público, por trimestre
Positivo = las cuotas le daban al local más chances de las que tuvo. Gráfico sugerido: barras, X = `trimestre`, Y = `error_mercado_pp`.

In [0]:
SELECT trimestre,
       sum(partidos) AS partidos,
       round(sum(error_mercado_local_pp * partidos) / sum(partidos), 2) AS error_mercado_pp
FROM workspace.tp_futbol.gold_mercado_trimestral
WHERE regimen_publico = 'sin_publico'
GROUP BY trimestre
HAVING sum(partidos) >= 30
ORDER BY trimestre;

trimestre,partidos,error_mercado_pp
2020-04-01,187,4.62
2020-07-01,281,-0.9
2020-10-01,549,7.91
2021-01-01,765,0.45
2021-04-01,500,-1.78
2021-07-01,169,-2.04


## 3. Margen de cada casa de apuestas por liga (%)
Gráfico sugerido: barras agrupadas, X = `liga`, Y = las tres columnas de margen.

In [0]:
SELECT liga,
       round(avg(margen_pinnacle) * 100, 2)          AS margen_pinnacle_pct,
       round(avg(margen_promedio_mercado) * 100, 2)  AS margen_promedio_mercado_pct,
       round(avg(margen_b365) * 100, 2)              AS margen_b365_pct,
       count(margen_b365)                            AS partidos_con_b365
FROM workspace.tp_futbol.gold_partidos
GROUP BY liga
ORDER BY liga;

liga,margen_pinnacle_pct,margen_promedio_mercado_pct,margen_b365_pct,partidos_con_b365
ARG,3.18,7.09,7.04,643
D1,2.71,4.79,5.62,2196
E0,2.59,4.31,5.48,2730
F1,2.67,5.05,5.64,2399
I1,2.7,4.91,5.63,2730
SP1,2.68,4.79,5.57,2748


## 4. Evolución del margen de Pinnacle por año (%)
Gráfico sugerido: líneas, X = `anio`, Y = `margen_pinnacle_pct`, una serie por `liga`.

In [0]:
SELECT liga, anio,
       count(margen_pinnacle)                  AS partidos,
       round(avg(margen_pinnacle) * 100, 2)    AS margen_pinnacle_pct
FROM workspace.tp_futbol.gold_partidos
WHERE margen_pinnacle IS NOT NULL
GROUP BY liga, anio
ORDER BY liga, anio;

liga,anio,partidos,margen_pinnacle_pct
ARG,2012,188,2.56
ARG,2013,383,2.67
ARG,2014,382,2.54
ARG,2015,469,2.53
ARG,2016,451,2.52
ARG,2017,406,2.51
ARG,2018,397,2.95
ARG,2019,328,3.48
ARG,2020,191,4.36
ARG,2021,525,3.74


## 5. Calibración de las cuotas y sesgo favorito-batacazo
Cada partido aporta tres apuestas (local, empate, visitante) con la cuota principal.
`resultado_por_peso_pct` = ganancia o pérdida promedio por cada $1 apostado. Gráfico sugerido: barras, X = `tramo`, Y = `resultado_por_peso_pct`.

In [0]:
WITH apuestas AS (
  SELECT cuota_local AS cuota, CASE WHEN resultado = 'H' THEN 1 ELSE 0 END AS gana
  FROM workspace.tp_futbol.gold_partidos WHERE cuota_local IS NOT NULL
  UNION ALL
  SELECT cuota_empate, CASE WHEN resultado = 'D' THEN 1 ELSE 0 END
  FROM workspace.tp_futbol.gold_partidos WHERE cuota_empate IS NOT NULL
  UNION ALL
  SELECT cuota_visitante, CASE WHEN resultado = 'A' THEN 1 ELSE 0 END
  FROM workspace.tp_futbol.gold_partidos WHERE cuota_visitante IS NOT NULL
)
SELECT
  CASE WHEN 1/cuota <= 0.15 THEN '1) 0-15%'
       WHEN 1/cuota <= 0.25 THEN '2) 15-25%'
       WHEN 1/cuota <= 0.35 THEN '3) 25-35%'
       WHEN 1/cuota <= 0.45 THEN '4) 35-45%'
       WHEN 1/cuota <= 0.55 THEN '5) 45-55%'
       WHEN 1/cuota <= 0.65 THEN '6) 55-65%'
       WHEN 1/cuota <= 0.75 THEN '7) 65-75%'
       ELSE '8) 75-100%' END              AS tramo,
  count(*)                                 AS apuestas,
  round(avg(1/cuota) * 100, 1)             AS prob_implicita_pct,
  round(avg(gana) * 100, 1)                AS frecuencia_real_pct,
  round((avg(gana * cuota) - 1) * 100, 1)  AS resultado_por_peso_pct
FROM apuestas
GROUP BY 1
ORDER BY 1;

tramo,apuestas,prob_implicita_pct,frecuencia_real_pct,resultado_por_peso_pct
1) 0-15%,5181,10.8,10.1,-6.5
2) 15-25%,13324,20.7,19.2,-7.5
3) 25-35%,25944,29.8,29.1,-2.4
4) 35-45%,9330,39.7,38.2,-3.9
5) 45-55%,6463,49.6,48.9,-1.5
6) 55-65%,4220,59.6,58.4,-2.1
7) 65-75%,2405,69.5,69.9,0.3
8) 75-100%,1779,81.6,78.8,-3.6


## 6. ¿Alguna estrategia simple le gana a la casa?
Resultado por cada $1 apostado (%), con la cuota principal y con Bet365.

In [0]:
WITH p AS (
  SELECT *,
         least(cuota_local, cuota_empate, cuota_visitante) AS cuota_favorito,
         CASE least(cuota_local, cuota_empate, cuota_visitante)
              WHEN cuota_local THEN 'H' WHEN cuota_empate THEN 'D' ELSE 'A' END AS pick_favorito,
         greatest(cuota_local, cuota_visitante) AS cuota_batacazo,
         CASE WHEN cuota_local >= cuota_visitante THEN 'H' ELSE 'A' END AS pick_batacazo
  FROM workspace.tp_futbol.gold_partidos
  WHERE cuota_local IS NOT NULL
)
SELECT 'siempre_local' AS estrategia,
       round((avg(CASE WHEN resultado = 'H' THEN cuota_local ELSE 0 END) - 1) * 100, 2) AS cuota_principal_pct,
       round((avg(CASE WHEN resultado = 'H' THEN b365c_local ELSE 0 END) FILTER (WHERE b365c_local IS NOT NULL) - 1) * 100, 2) AS bet365_pct
FROM p
UNION ALL
SELECT 'siempre_empate',
       round((avg(CASE WHEN resultado = 'D' THEN cuota_empate ELSE 0 END) - 1) * 100, 2),
       round((avg(CASE WHEN resultado = 'D' THEN b365c_empate ELSE 0 END) FILTER (WHERE b365c_empate IS NOT NULL) - 1) * 100, 2)
FROM p
UNION ALL
SELECT 'siempre_visitante',
       round((avg(CASE WHEN resultado = 'A' THEN cuota_visitante ELSE 0 END) - 1) * 100, 2),
       round((avg(CASE WHEN resultado = 'A' THEN b365c_visitante ELSE 0 END) FILTER (WHERE b365c_visitante IS NOT NULL) - 1) * 100, 2)
FROM p
UNION ALL
SELECT 'siempre_favorito',
       round((avg(CASE WHEN resultado = pick_favorito THEN cuota_favorito ELSE 0 END) - 1) * 100, 2), NULL
FROM p
UNION ALL
SELECT 'siempre_batacazo',
       round((avg(CASE WHEN resultado = pick_batacazo THEN cuota_batacazo ELSE 0 END) - 1) * 100, 2), NULL
FROM p;

estrategia,cuota_principal_pct,bet365_pct
siempre_local,-4.25,-8.26
siempre_empate,-2.38,-4.45
siempre_visitante,-4.6,-6.97
siempre_favorito,-2.28,null
siempre_batacazo,-6.56,null


## 7. ¿Influye la distancia que viaja el visitante?
Gráfico sugerido: barras, X = `tramo_distancia`, Y = `pct_local`, una serie por `region`.

In [0]:
SELECT CASE WHEN liga = 'ARG' THEN 'Argentina' ELSE 'Europa' END AS region,
       CASE WHEN distancia_km < 50   THEN '1) < 50 km'
            WHEN distancia_km < 200  THEN '2) 50-200 km'
            WHEN distancia_km < 500  THEN '3) 200-500 km'
            WHEN distancia_km < 1000 THEN '4) 500-1000 km'
            ELSE '5) > 1000 km' END                          AS tramo_distancia,
       count(*)                                              AS partidos,
       round(avg(CASE WHEN resultado = 'H' THEN 1 ELSE 0 END) * 100, 1) AS pct_local,
       round(avg(diferencia_goles), 3)                       AS diferencia_goles_promedio
FROM workspace.tp_futbol.gold_partidos
WHERE regimen_publico = 'pleno'
GROUP BY 1, 2
ORDER BY 1, 2;

region,tramo_distancia,partidos,pct_local,diferencia_goles_promedio
Argentina,1) < 50 km,1696,41.2,0.257
Argentina,2) 50-200 km,345,44.6,0.313
Argentina,3) 200-500 km,1659,45.2,0.333
Argentina,4) 500-1000 km,1764,43.7,0.325
Argentina,5) > 1000 km,394,42.6,0.305
Europa,1) < 50 km,839,43.1,0.26
Europa,2) 50-200 km,3125,43.0,0.244
Europa,3) 200-500 km,6305,44.4,0.33
Europa,4) 500-1000 km,3094,44.9,0.328
Europa,5) > 1000 km,171,46.2,0.322


## 8. Auditoría del pipeline (idempotencia)

In [0]:
SELECT * FROM workspace.tp_futbol.validacion_auditoria ORDER BY ejecutado_en;

ejecutado_en,bronze_partidos,silver_partidos,cuarentena_partidos,gold_partidos,controles_fallidos
2026-10-25T20:23:22.268Z,22887,22884,3,22884,0
2026-10-25T20:33:55.522Z,22887,22884,3,22884,0
2026-10-25T20:48:40.603Z,22887,22884,3,22884,0
2026-11-02T01:42:54.084Z,22887,22884,3,22884,0
2026-11-02T01:44:57.305Z,22887,22884,3,22884,0
